# 行列选择与条件筛选

学习目标：按标签、位置和条件选取小表中的记录，解释结果的顺序与形状，并完成可复现采样。

前置知识：Python 切片、布尔表达式、列表、Series 与 DataFrame、行列标签。

运行环境：Python 3.12、pandas 3.0；本章采用 pandas 3.0.6 的采样参数规则。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例均在单元内提供数据；后续单元沿用已导入的 pd。主线使用行列标签唯一、无缺失值的小表。

## 1 按成绩筛选记录

要找出成绩至少为 80 分的学生，先比较 score 列，得到每行对应一个 True 或 False 的布尔 Series，再把它放进表格的方括号中。True 对应的行被保留，原行标签与行顺序也保留。

下面的 students 是本章选择操作使用的原始表，行标签表示学生编号。

In [1]:
import pandas as pd

students = pd.DataFrame(
    {"name": ["小林", "小陈", "小周", "小赵"],
     "group": ["A", "B", "A", "B"], "score": [72, 90, 85, 60]},
    index=["S1", "S2", "S3", "S4"],
)
passed = students["score"] >= 80
print(students)  # 四行三列；输入顺序固定为 S1、S2、S3、S4。
print(passed)  # False、True、True、False，dtype 为 bool。
print(students[passed])  # 保留 S2、S3 的全部三列。
print(students[passed].shape)  # 预期：(2, 3)。

   name group  score
S1   小林     A     72
S2   小陈     B     90
S3   小周     A     85
S4   小赵     B     60
S1    False
S2     True
S3     True
S4    False
Name: score, dtype: bool
   name group  score
S2   小陈     B     90
S3   小周     A     85
(2, 3)


## 2 列选择与行列同时选择

列名唯一时，表格方括号中放一个列名得到 Series，放列名列表得到 DataFrame。列表同时决定输出列的顺序。

loc[行选择, 列选择] 按标签同时选行和列，冒号表示该轴全部内容。下面继续使用 students，按指定次序交付两名学生的成绩。

In [2]:
print(students["score"].shape)  # 预期：(4,)，一维 Series。
print(students[["score"]].shape)  # 预期：(4, 1)，二维 DataFrame。
selected = students.loc[["S3", "S1"], ["score", "name"]]
print(selected)  # 行依次 S3、S1；列依次 score、name；成绩为 85、72。
print(selected.shape)  # 预期：(2, 2)。
print(students.loc[passed, ["name", "score"]])  # 沿用 passed，结果为 S2、S3。

(4,)
(4, 1)
    score name
S3     85   小周
S1     72   小林
(2, 2)
   name  score
S2   小陈     90
S3   小周     85


## 3 标签切片与位置切片

loc 的切片使用标签，起止标签存在时包含两端；iloc 使用从 0 开始的位置，切片不包含终点。两者都不会因为选择操作自动重排原表。

下面继续使用按 S1 至 S4 排列的 students。表格的整数切片 students[1:3] 也选行；需要明确行列含义时用 iloc。标签切片不等同于按标签值做大小比较，本例使用唯一且有序的标签。

In [3]:
print(students.loc["S2":"S3", "name":"score"])
# S2、S3；列包含 name、group、score，两端都保留。
print(students.iloc[1:3, [2, 0]])
# 同样取 S2、S3；位置 2、0 对应 score、name。
print(students[1:3].index.tolist())  # 预期：['S2', 'S3']。
print(students.iloc[-1, 2])  # 预期：60，负位置 -1 表示最后一行。

   name group  score
S2   小陈     B     90
S3   小周     A     85
    score name
S2     90   小陈
S3     85   小周
['S2', 'S3']
60


### 3.1 整数标签仍是标签

标签也可以是整数。loc[0] 查找标签 0，iloc[0] 选当前第一行；默认整数索引有时让两者恰好一致，但不能据此混用。

pandas 3 中，Series[整数] 的单个整数键也始终按标签解释，不再回退为位置。下面单独创建一个整数标签顺序与位置不同的 Series。

In [4]:
scores = pd.Series([90, 70, 80], index=[20, 0, 10], name="score")
print(scores.loc[0], scores.iloc[0], scores[0])  # 预期：70 90 70。
print(scores.loc[20:0])  # 沿当前顺序包含标签 20、0，并非数值区间。
print(scores.iloc[0:1])  # 只含位置 0，即标签 20。

named = pd.Series([90, 70], index=["A", "B"])
try:
    named[0]
except KeyError:
    print("KeyError：标签 0 不存在")
else:
    raise AssertionError("预期按标签读取失败")

70 90 70
20    90
0     70
Name: score, dtype: int64
20    90
Name: score, dtype: int64
KeyError：标签 0 不存在


### 3.2 缺失标签与位置越界

按单个标签或标签列表读取时，所需标签不存在会报 KeyError；按单个位置读取时越界会报 IndexError。位置切片允许越过末端，只返回实际覆盖的部分。

下面沿用 students。标签切片的端点规则另有条件，不能把“列表中缺标签会报错”套用到所有标签切片。

In [5]:
try:
    students.loc[["S1", "S9"], ["score"]]
except KeyError:
    print("KeyError：S9 不存在")
else:
    raise AssertionError("预期缺失标签报错")

try:
    students.iloc[4]
except IndexError:
    print("IndexError：只有位置 0 至 3")
else:
    raise AssertionError("预期位置越界报错")

print(students.iloc[3:9].index.tolist())  # 预期：['S4']。
print(students.iloc[9:12].shape)  # 预期：(0, 3)，切片可返回空表。

KeyError：S9 不存在
IndexError：只有位置 0 至 3
['S4']
(0, 3)


## 4 单个值的读取

只需一个值时，at[行标签, 列标签] 使用标签，iat[行位置, 列位置] 使用位置。这里的标签均唯一，因此读取结果是标量。

下面沿用 students，比较读取 S2 成绩的写法。读取不存在的标签或越界位置仍会分别报 KeyError、IndexError。

In [6]:
print(students.at["S2", "score"])  # 预期：90。
print(students.iat[1, 2])  # 预期：90。
print(students.loc["S2", "score"])  # 两个轴均取单个标签，也得到 90。
print(students.loc[["S2"], ["score"]].shape)  # 预期：(1, 1)，列表保留二维结构。

90
90
90
(1, 1)


## 5 复合条件与空结果

isin 检查值是否属于给定集合，参数可用列表；即使只有一个字符串也应写成单元素列表。between 检查数值区间，inclusive 可设为 both、left、right、neither，分别表示包含两端、仅左端、仅右端或都不包含。

布尔 Series 用 & 表示逐元素“且”、| 表示“或”、~ 表示“非”；每个比较条件加括号，不用 Python 的 and、or 代替逐元素运算。下面沿用 students。

In [7]:
in_group = students["group"].isin(["A"])
in_range = students["score"].between(72, 85, inclusive="both")
print(students.loc[in_group & in_range, ["name", "score"]])
# S1、S3；72 和 85 两个端点都包含。
print(students.loc[students["score"].between(72, 85, inclusive="left")].index.tolist())
# 预期：['S1']，右端点 85 不包含。
print(students.loc[(students["score"] >= 90) | ~in_group].index.tolist())
# 预期：['S2', 'S4']。

empty = students.loc[students["score"] > 100, ["name", "score"]]
print(empty)  # 无符合条件的行，仍保留 name、score 两列。
print(empty.shape)  # 预期：(0, 2)。

   name  score
S1   小林     72
S3   小周     85
['S1']
['S2', 'S4']
Empty DataFrame
Columns: [name, score]
Index: []
(0, 2)


## 6 布尔 Series 按标签对齐

loc 接收可对齐的布尔 Series：先根据行标签匹配条件，再按原表的行顺序保留 True 对应的行。以下限定表和掩码的标签都唯一，掩码覆盖每个待选行标签；掩码可以乱序，多余标签不产生额外行。

下面仍筛选 students，但条件单独提供，用来观察标签匹配。

In [8]:
keep = pd.Series(
    [True, False, True, False, True],
    index=["S3", "S1", "S2", "S4", "S9"],
)
aligned = students.loc[keep, ["score"]]
print(aligned)  # S2 为 90、S3 为 85；结果顺序来自 students。
print(aligned.index.tolist(), aligned.shape)  # ['S2', 'S3'] (2, 1)。

incomplete = pd.Series([True, False], index=["S1", "S2"])
try:
    students.loc[incomplete]
except pd.errors.IndexingError:
    print("IndexingError：掩码缺少 S3、S4，无法对齐")
else:
    raise AssertionError("预期不可对齐的掩码报错")

    score
S2     90
S3     85


['S2', 'S3'] (2, 1)
IndexingError：掩码缺少 S3、S4，无法对齐


### 6.1 按位置使用布尔值

本例字符串索引的布尔 Series 会被 iloc 拒绝，但可以接收长度与行数相同的布尔列表或数组。把 Series 转成数组会去掉标签，此后只能按当前位置解释条件。

下面用同一组乱序条件比较两种选择。不要为绕过对齐错误而直接去掉标签；先确认条件和数据的位置确实一一对应。

官方用户指南将布尔 Series 列为 iloc 不支持的输入。按标签筛选使用 loc；按位置筛选使用长度一致且顺序已确认的布尔列表或数组。

In [9]:
positional_keep = pd.Series(
    [True, False, True, False], index=["S3", "S1", "S2", "S4"]
)
print(students.loc[positional_keep].index.tolist())  # 预期：['S2', 'S3']。
print(students.iloc[positional_keep.to_numpy()].index.tolist())
# 预期：['S1', 'S3']，现在 True 位于位置 0、2。

try:
    students.iloc[positional_keep]
except ValueError:
    print("ValueError：本例字符串索引的布尔 Series 会被 iloc 拒绝")
else:
    raise AssertionError("预期 iloc 拒绝本例字符串索引的布尔 Series")

['S2', 'S3']
['S1', 'S3']
ValueError：本例字符串索引的布尔 Series 会被 iloc 拒绝


### 6.2 选学：文档与本版本实现的差异

官方用户指南说明 iloc 不接受布尔 Series，但 pandas 3.0.6 的实现会放行整数索引的布尔 Series，再进入标签对齐分支。下面的乱序整数标签示例用于观察这一差异，不把它当作受文档保证的用法，也不保证其他版本有相同行为。

按标签匹配条件时显式使用 loc；需要按当前位置解释条件时，确认顺序后再使用布尔数组。

In [10]:
values = pd.Series([10, 20, 30])
integer_mask = pd.Series([True, False, False], index=[2, 0, 1])
print(values.iloc[integer_mask].tolist())  # 本环境为 [30]，条件按标签对齐。
print(values.loc[integer_mask].tolist())  # [30]，明确表达标签筛选。
print(values.iloc[integer_mask.to_numpy()].tolist())  # [10]，去掉标签后按位置。
assert values.iloc[integer_mask].tolist() == [30]

[30]
[30]
[10]


## 7 随机采样

### 7.1 固定数量与比例

sample 默认抽取行。n 指定条数，frac 指定占原行数的比例，两者不能同时指定；默认 replace=False，表示无放回，同一行不会重复抽中。返回结果保留抽中行的原标签，顺序是采样结果的顺序。

下面沿用顺序固定为 S1、S2、S3、S4 的 students。相同输入顺序、参数和运行环境下，每次传同一个整数 random_state，可以重复得到同一结果；改变输入顺序后，不能只凭种子保证选中相同标签。

In [11]:
first = students.sample(n=2, replace=False, random_state=7)
again = students.sample(n=2, replace=False, random_state=7)
half = students.sample(frac=0.5, replace=False, random_state=7)
print(first)  # 本例依次抽出 S3、S2，保留原标签与三列。
print(first.equals(again))  # 预期：True，相同输入与整数种子可复现。
print(first.shape, first.index.is_unique)  # 预期：(2, 3) True。
print(half.shape)  # 预期：(2, 3)，四行的一半为两行。
assert first.equals(again)
assert set(first.index).issubset(students.index)

   name group  score
S3   小周     A     85
S2   小陈     B     90
True
(2, 3) True
(2, 3)


### 7.2 有放回与样本规模

无放回时，n 不能超过原行数；需要更多条记录时，replace=True 允许重复抽取。同理，frac 大于 1 时需要有放回。n=0 可得到零行样本。

下面从同一张四行的 students 抽六次；这时必然存在重复标签，但不代表原表有重复记录。

In [12]:
repeated = students.sample(n=6, replace=True, random_state=7)
print(repeated.index.tolist())  # 六个标签，来自 S1 至 S4，其中存在重复。
print(repeated.shape, repeated.index.is_unique)  # 预期：(6, 3) False。
print(students.sample(n=0, random_state=7).shape)  # 预期：(0, 3)。

# parameters 保存每组参数；**parameters 将字典展开为关键字参数。
for parameters in [{"n": 5}, {"n": 2, "frac": 0.5}, {"frac": 1.5}]:
    try:
        students.sample(**parameters, replace=False, random_state=7)
    except ValueError:
        print("ValueError：不兼容的采样参数", parameters)
    else:
        raise AssertionError("预期采样参数检查失败")
# 分别检验无放回超额、同时指定 n 和 frac、无放回比例超过 1。

['S4', 'S1', 'S2', 'S3', 'S4', 'S4']
(6, 3) False
(0, 3)
ValueError：不兼容的采样参数 {'n': 5}
ValueError：不兼容的采样参数 {'n': 2, 'frac': 0.5}
ValueError：不兼容的采样参数 {'frac': 1.5}


## 8 选学：保留原形状的条件替换

筛选会删去不符合条件的行；where 和 mask 用于保留原来的位置并替换值。where 在条件为 True 时保留，mask 在条件为 True 时替换；other 指定替代值。

下面使用独立的四个成绩，把不足 80 分的位置显示为 0；0 只是本例的展示约定。默认不传 other 会填入缺失值，结果类型需要另外检查。

In [13]:
display_scores = pd.Series([72, 90, 85, 60], index=["S1", "S2", "S3", "S4"])
kept = display_scores.where(display_scores >= 80, other=0)
replaced = display_scores.mask(display_scores < 80, other=0)
print(kept)  # 四个位置依次为 0、90、85、0；dtype 仍为 int64。
print(kept.equals(replaced))  # 预期：True，两种条件方向相反。
print(kept.shape, kept.index.equals(display_scores.index))  # (4,) True。
print(display_scores.tolist())  # 预期：[72, 90, 85, 60]，原数据未修改。

S1     0
S2    90
S3    85
S4     0
dtype: int64


True
(4,) True
[72, 90, 85, 60]


## 9 选学：加权采样

weights 指定相对权重，pandas 会归一化；权重不是最终样本必须满足的比例。权重 Series 按标签对齐，未提供权重的行视为权重 0，多余标签被忽略。权重应非负、有限且总和大于 0；缺失权重按 0 处理。

下面沿用 students，只有 S2、S3 有正权重。固定整数种子，用有放回采样观察零权重行不会被抽中。

In [14]:
weights = pd.Series([3.0, 1.0], index=["S3", "S2"])
weighted = students.sample(n=4, replace=True, weights=weights, random_state=7)
print(weighted.index.tolist())  # 四次抽样只可能出现 S2、S3。
print(set(weighted.index).issubset({"S2", "S3"}))  # 预期：True。
assert set(weighted.index).issubset({"S2", "S3"})

['S2', 'S3', 'S3', 'S3']
True


pandas 3.0.6 的无放回加权采样还会检查 n × 最大权重 ÷ 权重总和是否大于 1；超过时会拒绝采样，以避免偏差。这里 n 是请求条数，不是原表行数。

下面选两行，而归一化最大权重为 0.75，乘积为 1.5，因此不能直接使用这组权重完成无放回采样。应按任务重新考虑抽样方案，不应悄悄改成有放回。

In [15]:
try:
    students.sample(n=2, replace=False, weights=weights, random_state=7)
except ValueError:
    print("ValueError：无放回加权采样超出权重限制")
else:
    raise AssertionError("预期权重限制报错")

ValueError：无放回加权采样超出权重限制


## 10 选学：用表达式筛选

query 可以把涉及列名的布尔筛选写成字符串，适合固定、可读的条件。下面沿用 students，明确使用 Python 计算引擎，不需要可选的 NumExpr。

query 可能执行任意代码，只应使用自己编写或已审查的可信表达式，不能直接拼接外部输入。它是筛选表达式接口，不是处理不可信查询的安全边界。

In [16]:
queried = students.query("score >= 80 and group == 'A'", engine="python")
ordinary = students.loc[(students["score"] >= 80) & (students["group"] == "A")]
print(queried)  # 仅 S3，保留全部三列。
print(queried.equals(ordinary))  # 预期：True。
# query 的表达式语法允许这里的 and；普通 Series 条件仍使用 &。

   name group  score
S3   小周     A     85
True


## 本章小结

（1）先确定按标签、位置还是条件选择；loc 与 iloc 的整数含义和切片终点不同。

（2）行列选择不仅要核对值，还要检查标签、顺序和结果维数；空结果仍可保留列结构。

（3）可对齐的布尔 Series 按标签匹配，布尔数组按位置解释；去掉标签会改变语义。

（4）采样先确定数量、比例和是否放回；复现需要相同原始输入顺序、参数和环境。

（5）选学的 where、mask 保留形状并替换值；query 只使用可信表达式。

## 练习

（1）先预测下面三个结果的标签与数值，再运行核对。说明整数标签与整数位置分别在哪里生效。

In [17]:
exercise_scores = pd.Series([60, 80, 90], index=[2, 0, 1])
print(exercise_scores.loc[2:0])
print(exercise_scores.iloc[0:2])
print(exercise_scores[0])
# 运行前记录预测；运行后核对切片两端与单个整数键的含义。

2    60
0    80
dtype: int64
2    60
0    80
dtype: int64
80


（2）筛出 A 组中成绩在 70 至 90 分之间的学生，两端都包含；结果只保留 name、score 两列，并保持原行顺序。再把范围改为 95 至 100 分，说明空结果的形状。

In [18]:
exercise_table = pd.DataFrame(
    {"name": ["甲", "乙", "丙", "丁"], "group": ["A", "B", "A", "A"],
     "score": [70, 85, 90, 60]}, index=["R1", "R2", "R3", "R4"],
)
# 在此使用 isin、between 和 loc 完成筛选。
# 检查：第一份结果为 R1、R3，列依次 name、score，shape 为 (2, 2)。
# 第二份结果应保留相同列结构；说明 inclusive 参数如何表达端点要求。

（3）下面的批准名单乱序提供。任务要求“按记录编号匹配”，应选择 loc 还是把条件转成数组交给 iloc？写出结果并说明理由。

然后改变条件：外部系统只提供按当前行顺序排列的布尔列表。写出适合的选择方式，并解释为什么此时必须检查列表长度与顺序。

In [19]:
records = pd.DataFrame({"amount_yuan": [10, 20, 30]}, index=["A", "B", "C"])
approved = pd.Series([True, False, True], index=["C", "A", "B"])
position_flags = [True, False, True]
# 在此分别按记录编号和按当前位置完成选择，并用注释解释理由。
# 检查：标签匹配结果为 B、C；位置条件结果为 A、C；金额单位为元。
# 若 approved 缺少 B，应报告无法对齐，不能直接删掉标签绕过检查。

（4）从下面四条记录中无放回抽两条，固定输入顺序和 random_state，重复运行并比较结果。随后把需求改成抽六次，允许同一条记录多次出现：应改变哪个参数？说明理由，并检查样本数和重复标签。

In [20]:
population = pd.DataFrame({"value": [10, 20, 30, 40]}, index=["A", "B", "C", "D"])
seed = 11
# 在此完成两次可复现的无放回采样，再完成六次有放回采样。
# 检查：两份两行样本 equals 为 True；每份两行样本标签唯一。
# 六次抽样保留原标签，结果有六行且存在重复标签；不要同时指定 n 和 frac。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) 的 Basics、Slicing ranges、Selection by label、Selection by position、Fast scalar value getting and setting、Boolean indexing：方括号、切片、复合条件及布尔数组；[loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) 的 Allowed inputs、Raises、Examples：标签与可对齐布尔 Series；[iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) 的位置选择、越界与切片；[at](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.at.html)、[iat](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iat.html) 的标量读取及异常；[pandas 3.0.0 release notes](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html) 的 Other Removals：Series 整数键始终按标签；[Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) 的成员检查与单字符串限制；[Series.between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) 的 inclusive；[DataFrame.sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) 的 n、frac、replace、weights、random_state 及 Notes 中无放回权重限制（权重非负与非零总和见下列固定版本源码）；[Series.where](https://pandas.pydata.org/docs/reference/api/pandas.Series.where.html)、[Series.mask](https://pandas.pydata.org/docs/reference/api/pandas.Series.mask.html) 的条件方向、other 与返回形状；[DataFrame.query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) 的表达式、engine、Notes 及代码注入警告。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[indexing](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/indexing.rst)、[v3.0.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.0.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 [indexing.py](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L1592-L1600) 的 _iLocIndexer._validate_key、[第 1222 行](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L1222-L1227) 的 _getbool_axis 与 [check_bool_indexer](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/indexing.py#L2647)：整数索引布尔 Series 的放行及标签对齐，与指南 Boolean indexing 的限制不一致，仅说明本版本实现。[sample.py](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/sample.py) 的 preprocess_weights、sample：负权重与零权重总和的检查。 |